In [1]:
import sqlite3
import pandas as pd
import os
from pybaseball import statcast
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../data/statcast_2026.db'
os.makedirs('../data', exist_ok=True)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
print("Database connected: " + DB_PATH)

Database connected: ../data/statcast_2026.db


In [2]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS pitches (
    game_date TEXT,
    pitcher INTEGER,
    pitcher_name TEXT,
    batter INTEGER,
    batter_name TEXT,
    events TEXT,
    description TEXT,
    zone INTEGER,
    stand TEXT,
    p_throws TEXT,
    home_team TEXT,
    away_team TEXT,
    inning INTEGER,
    inning_topbot TEXT,
    pitch_type TEXT,
    pitch_name TEXT,
    release_speed REAL,
    release_pos_x REAL,
    release_pos_z REAL,
    pfx_x REAL,
    pfx_z REAL,
    plate_x REAL,
    plate_z REAL,
    vx0 REAL,
    vy0 REAL,
    vz0 REAL,
    ax REAL,
    ay REAL,
    az REAL,
    effective_speed REAL,
    release_spin_rate REAL,
    release_extension REAL,
    spin_axis REAL,
    launch_speed REAL,
    launch_angle REAL,
    hit_distance_sc REAL,
    hc_x REAL,
    hc_y REAL,
    bb_type TEXT,
    balls INTEGER,
    strikes INTEGER,
    on_1b INTEGER,
    on_2b INTEGER,
    on_3b INTEGER,
    outs_when_up INTEGER,
    at_bat_number INTEGER,
    pitch_number INTEGER,
    delta_home_win_exp REAL,
    delta_run_exp REAL,
    bat_score INTEGER,
    fld_score INTEGER,
    estimated_ba_using_speedangle REAL,
    estimated_woba_using_speedangle REAL,
    woba_value REAL,
    iso_value REAL,
    launch_speed_angle REAL
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS games (
    game_date TEXT PRIMARY KEY,
    home_team TEXT,
    away_team TEXT,
    home_score INTEGER,
    away_score INTEGER,
    total_pitches INTEGER,
    jays_pitches INTEGER,
    jays_batting_pitches INTEGER,
    jays_starter TEXT,
    jays_home_away TEXT
)
''')

conn.commit()
print("Tables created: pitches, games")

Tables created: pitches, games


In [3]:
COLUMNS_TO_KEEP = [
    'game_date', 'pitcher', 'player_name', 'batter', 'stand', 'p_throws',
    'home_team', 'away_team', 'inning', 'inning_topbot',
    'pitch_type', 'pitch_name', 'events', 'description', 'zone',
    'release_speed', 'release_pos_x', 'release_pos_z',
    'pfx_x', 'pfx_z', 'plate_x', 'plate_z',
    'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az',
    'effective_speed', 'release_spin_rate', 'release_extension', 'spin_axis',
    'launch_speed', 'launch_angle', 'hit_distance_sc', 'hc_x', 'hc_y', 'bb_type',
    'balls', 'strikes', 'on_1b', 'on_2b', 'on_3b', 'outs_when_up',
    'at_bat_number', 'pitch_number',
    'delta_home_win_exp', 'delta_run_exp',
    'bat_score', 'fld_score',
    'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle',
    'woba_value', 'iso_value', 'launch_speed_angle'
]

def load_game_to_db(game_date, conn):
    existing = pd.read_sql_query(
        "SELECT COUNT(*) as cnt FROM pitches WHERE game_date = ?",
        conn, params=[game_date]
    )
    if existing['cnt'].iloc[0] > 0:
        print("  " + game_date + " already in DB — skipping")
        return 0

    data = statcast(start_dt=game_date, end_dt=game_date)

    if len(data) == 0:
        print("  " + game_date + " — no data found")
        return 0

    tor_games = data[(data['home_team'] == 'TOR') | (data['away_team'] == 'TOR')]
    if len(tor_games) == 0:
        print("  " + game_date + " — no Blue Jays game")
        return 0

    available_cols = [c for c in COLUMNS_TO_KEEP if c in tor_games.columns]
    tor_games = tor_games[available_cols].copy()

    rename_map = {'player_name': 'pitcher_name'}
    tor_games = tor_games.rename(columns=rename_map)

    tor_games['game_date'] = game_date

    tor_games.to_sql('pitches', conn, if_exists='append', index=False)

    if 'TOR' in tor_games['home_team'].values:
        jays_home_away = 'HOME'
        jays_pitching = tor_games[tor_games['inning_topbot'] == 'Top']
        jays_batting = tor_games[tor_games['inning_topbot'] == 'Bot']
    else:
        jays_home_away = 'AWAY'
        jays_pitching = tor_games[tor_games['inning_topbot'] == 'Bot']
        jays_batting = tor_games[tor_games['inning_topbot'] == 'Top']

    home_score = tor_games['bat_score'].max() if jays_home_away == 'HOME' else tor_games['fld_score'].max()
    away_score = tor_games['fld_score'].max() if jays_home_away == 'HOME' else tor_games['bat_score'].max()

    starter_name = ''
    if len(jays_pitching) > 0:
        first_inning = jays_pitching[jays_pitching['inning'] == jays_pitching['inning'].min()]
        if len(first_inning) > 0:
            starter_name = first_inning['pitcher_name'].mode().iloc[0] if len(first_inning['pitcher_name'].mode()) > 0 else ''

    game_info = pd.DataFrame([{
        'game_date': game_date,
        'home_team': tor_games['home_team'].iloc[0],
        'away_team': tor_games['away_team'].iloc[0],
        'home_score': int(home_score) if pd.notna(home_score) else 0,
        'away_score': int(away_score) if pd.notna(away_score) else 0,
        'total_pitches': len(tor_games),
        'jays_pitches': len(jays_pitching),
        'jays_batting_pitches': len(jays_batting),
        'jays_starter': starter_name,
        'jays_home_away': jays_home_away
    }])
    game_info.to_sql('games', conn, if_exists='append', index=False)

    print("  " + game_date + " — " + str(len(tor_games)) + " pitches loaded (starter: " + starter_name + ")")
    return len(tor_games)

In [4]:
from datetime import datetime, timedelta

start_date = datetime(2026, 3, 26)
end_date = datetime(2026, 5, 25)

total_loaded = 0
games_loaded = 0

current = start_date
while current <= end_date:
    date_str = current.strftime('%Y-%m-%d')
    try:
        count = load_game_to_db(date_str, conn)
        if count > 0:
            games_loaded += 1
            total_loaded += count
    except Exception as e:
        print("  " + date_str + " — ERROR: " + str(e))
    current += timedelta(days=1)

conn.commit()
print("\n=== LOAD COMPLETE ===")
print("Games loaded: " + str(games_loaded))
print("Total pitches: " + str(total_loaded))

This is a large query, it may take a moment to complete


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  2.16it/s]

  2026-03-26 — no Blue Jays game
This is a large query, it may take a moment to complete



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  4.10it/s]

  2026-03-27 — 265 pitches loaded (starter: Gausman, Kevin)
This is a large query, it may take a moment to complete


...

  2026-05-23 — 267 pitches loaded (starter: Corbin, Patrick)
This is a large query, it may take a moment to complete



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  2.06it/s]

  2026-05-24 — 298 pitches loaded (starter: Cease, Dylan)
This is a large query, it may take a moment to complete



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  2.91it/s]

  2026-05-25 — 303 pitches loaded (starter: Yesavage, Trey)

=== LOAD COMPLETE ===
Games loaded: 54
Total pitches: 15614


In [5]:
print("=== DATABASE SUMMARY ===\n")

game_count = pd.read_sql_query("SELECT COUNT(*) as cnt FROM games", conn)
print("Total games: " + str(game_count['cnt'].iloc[0]))

pitch_count = pd.read_sql_query("SELECT COUNT(*) as cnt FROM pitches", conn)
print("Total pitches: " + str(pitch_count['cnt'].iloc[0]))

date_range = pd.read_sql_query("SELECT MIN(game_date) as first, MAX(game_date) as last FROM games", conn)
print("Date range: " + str(date_range['first'].iloc[0]) + " to " + str(date_range['last'].iloc[0]))

print("\n=== GAMES LIST ===")
games = pd.read_sql_query("""
    SELECT game_date, home_team, away_team, home_score, away_score,
           jays_starter, jays_home_away, total_pitches
    FROM games
    ORDER BY game_date
""", conn)
print(games.to_string(index=False))

print("\n=== PITCHERS SUMMARY ===")
pitchers = pd.read_sql_query("""
    SELECT pitcher_name, COUNT(*) as total_pitches, COUNT(DISTINCT game_date) as games
    FROM pitches
    WHERE (home_team = 'TOR' AND inning_topbot = 'Top')
       OR (away_team = 'TOR' AND inning_topbot = 'Bot')
    GROUP BY pitcher_name
    ORDER BY total_pitches DESC
    LIMIT 20
""", conn)
print(pitchers.to_string(index=False))

=== DATABASE SUMMARY ===

Total games: 54
Total pitches: 15614
Date range: 2026-03-27 to 2026-05-25

=== GAMES LIST ===
 game_date home_team away_team  home_score  away_score    jays_starter jays_home_away  total_pitches
2026-03-27       TOR       ATH           2           2  Gausman, Kevin           HOME            265
2026-03-28       TOR       ATH           7           7    Cease, Dylan           HOME            366
2026-03-29       TOR       ATH           5           5     Lauer, Eric           HOME            250
2026-03-30       TOR       COL          14          14     Ponce, Cody           HOME            333
2026-03-31       TOR       COL           5           5   Scherzer, Max           HOME            301
2026-04-01       TOR       COL           2           2  Gausman, Kevin           HOME            320
2026-04-03       CWS       TOR           4           4    Cease, Dylan           AWAY            317
2026-04-04       CWS       TOR           6           4 Fluharty, Mason  

In [6]:
def daily_update(game_date, conn):
    print("=== Daily Update: " + game_date + " ===")
    count = load_game_to_db(game_date, conn)
    conn.commit()
    if count > 0:
        print("Update complete: " + str(count) + " pitches added")
    else:
        print("No new data added")
    return count

In [7]:
conn.close()
print("Database connection closed.")
print("DB saved at: " + DB_PATH)

Database connection closed.
DB saved at: ../data/statcast_2026.db


In [8]:
import sqlite3

conn = sqlite3.connect(DB_PATH)

daily_update('2026-05-28', conn)
daily_update('2026-05-29', conn)
daily_update('2026-05-30', conn)
daily_update('2026-05-31', conn)

conn.close()
print("BAL series data added!")

=== Daily Update: 2026-05-28 ===
This is a large query, it may take a moment to complete


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  3.61it/s]

  2026-05-28 — 282 pitches loaded (starter: Corbin, Patrick)
Update complete: 282 pitches added
=== Daily Update: 2026-05-29 ===
This is a large query, it may take a moment to complete



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  2.26it/s]

  2026-05-29 — 278 pitches loaded (starter: Macko, Adam)
Update complete: 278 pitches added
=== Daily Update: 2026-05-30 ===
This is a large query, it may take a moment to complete



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  2.06it/s]

  2026-05-30 — 307 pitches loaded (starter: Yesavage, Trey)
Update complete: 307 pitches added
=== Daily Update: 2026-05-31 ===
This is a large query, it may take a moment to complete



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.95it/s]

  2026-05-31 — 284 pitches loaded (starter: Miles, Spencer)
Update complete: 284 pitches added
BAL series data added!
